[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourrepo/notebook.ipynb)

In [ ]:
# if using google collab, **ensure** that you run this code block to clone the repo first
!git clone https://github.com/DCS-training/Understanding-Creating-WordEmbeddings.git
%cd Understanding-Creating-WordEmbeddings

# Introduction to Topic Modelling with LDA
### Author: Dr. Somya Iqbal

Topic modelling is a computational text analysis method used to uncover **hidden thematic structures** in large collections of text.
- Instead of reading each document individually, topic models identify **patterns of word co-occurrence** across the corpus.
- These patterns form **topics**, which are groups of words that frequently appear together.
- Each document can contain **multiple topics in different proportions**, rather than belonging to only one theme.

In this workshop we will focus on **Latent Dirichlet Allocation (LDA)**, one of the most widely used topic modelling algorithms. The workflow will follow the general logic and protocols of the sister workflow in the R version of this subject (in this repo).

> Before we begin there are some key things to understand about this Python workflow and the sister workflow in R. You can start with reading this overview to understand the intricacies of replicating a workflow with applied method approaches that may differ across programming packages yet still remain equally valid. LDA (Latent Dirichlet Allocation) -> can be employed through a few different ways, starting with the foundational paper which introduced the method [paper](https://www.jmlr.org/papers/volume3/blei03a/blei03a.pdf). Between Python and R languages various authors and developers create and validate packages for models and many options are available. Implementation can differ in underlying mathematical approach or options for parameters and development with new knowledge in the field. Therefore, despite having the same task or goal of topic inference, the HOW of the model and route may differ which can create discrepancies between equivalent workflows across programming languages. A few ways to select which package to use for applying a model; you might assess conventions in your domain through literature, assess library documentation to assess whether your method scope is possible with a given package and suitability for your research question and workflow.

In the sister R workflow of this tutorial `TopicModels` was used for LDA [See more](https://cran.r-universe.dev/topicmodels/doc/manual.html)
In this Python workflow scikitlearn `LDA`was used [See here](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)

# 1) GETTING STARTED

## 1a. Installation of relevant packages

In [ ]:
%pip install numpy pandas matplotlib scikit-learn nltk plotly 
!pip install gensim pyLDAvis
# NLTK is composed of a natural language toolkit for Python. 
# numpy and pandas are used for dealing with numerical functions and also for tabular data in the data CSV.
# scikit-learn, taking two key tools from this library count vectoriser and dealing with stop-words.
# matplotlib facilitates visualisation for graphing in the notebook.
#plotly for interactive viz
#gensim to use coherence model (evaluation metric via score)
#pyLDAviz is a dedicated visualisation tool for exploring LDA fitted model/topics for exploring data

## 1b. Importing for setup
Getting relevant tools and features imported and in place for use in the codebook and their purpose

In [80]:
from pathlib import Path #helps to set out file/folder paths
import re # in text analysis we may want to use regular expressions for formulaically clearing up data (use re)
import numpy as np #numeric/matrix handling
import pandas as pd # dataframes and CSVs
import matplotlib.pyplot as plt #for the plots to come
import textwrap  # supports wrapping long text output for  better readability
import plotly.express as px
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS # build document-term matrix & cleaning filler words  like 'a', 'the' (English specific)
from sklearn.decomposition import LatentDirichletAllocation #LatentDirichletAllocation from sklearn (for fitting the topic model)
from nltk.stem import PorterStemmer # A stemmer reduces words to their root, "reading to read" for instance
from gensim.corpora import Dictionary #a requisite for steps before gensim coherence model can be used.
from gensim.models.coherencemodel import CoherenceModel #we want to use the gensim coherence score model
import pyLDAvis #we want to visualise coherence score metrics 

# note NLTK also has a built in stop words function 

## 1c. Setting an output folder for later steps

In [81]:
outputs_dir = Path("../outputs")
outputs_dir.mkdir(exist_ok=True)
# creates an output folder in the same directory

# 2) DATA OVERVIEW

Lets get to the know the data a little. Typically you will have curated a set of data or have a specific set for your research which you will be familiar with. Once it is imported in your workflow, it is good to inspect items like the dimensions, features, variables, and observations as a point of good practice.

## 2a. An inspection of the US inaugural speech dataset (overview)

In [ ]:
speeches_df = pd.read_csv("../data/inaugural_speeches.csv") # Loading and reading in CSV from data folder

print("Dimension:", speeches_df.shape) #overall shape of speech/text
print("\nColumn names:") #column names 
print(speeches_df.columns.tolist()) #showing the columns in list

display(speeches_df.head()) #showing a little preview

print("Summary:")
speeches_df.info()

- *Note in the output print showing a brief view of the data. The Info table (summary) then presents a summary of the data columns, variables counts start at 0, 1, 2, 3 totalling 4 columns and 60 entries across them. There are no missing values in the dataframe*

## 2b. Quick inspection of the speech content
#### What's available in the corpus?

In [ ]:
display(speeches_df.head())

In [ ]:
# inspect the speech text first 20 rows in the series, snapshot
print("speech from text column:")
print(speeches_df.loc[0:20, "text"][:1000]) # python starts at 0 when iterating through items.

print("first speech:") #first speech detail view
print(textwrap.fill(speeches_df.loc[0, "text"][:1000], width=100))

## 3a. Getting data ready for pre-processing and tidy steps

In [85]:
speech_texts = speeches_df["text"].astype(str).tolist()  # extract the speech texts into a plain Python list

## 3b. Visualising the data at current stages

A brief summary of words & characters in the corpus

In [ ]:
speeches_df["character_count"] = speeches_df["text"].astype(str).str.len()
speeches_df["word_count"] = speeches_df["text"].astype(str).str.split().str.len()

display(
    speeches_df[
        ["document", "year", "president", "character_count", "word_count"]
    ].head()
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(speeches_df["year"], speeches_df["word_count"], marker="o")
plt.xlabel("Year")
plt.ylabel("Word count")
plt.title("Speech length over time")
plt.show()

# 4) DATA HANDLING

## 4a. Simple data handling (text focussed)

In [ ]:
def basic_clean_text(text): # Set basic cleaning function to standardise the text.
    text = text.lower() # convert text to lowercase, removes line breaks
    text = text.replace("\n", " ") 
    text = re.sub(r"\s+", " ", text).strip() #whitespace handling > collapse repeated spaces into a single space (we standardise).
    return text

# Apply cleaning to every speech
cleaned_speech_texts = [basic_clean_text(text) for text in speech_texts]

print("cleaned 1st speech:\n")
print(textwrap.fill(cleaned_speech_texts[0][:500], width=100)) #wrapping the text for less scrolling and readability

#### Things look in order for basic cleaning with items in lowercase and relative uniformity. Next steps we would want conduct some  pre-processing of contents a little more vigorously such as dealing with punctuation and stop words. However before proceeding with data pre-processing we need to make some considerations. It may not be visible yet, because these steps have simply standardised items for later where we will want to extract words using the uniform whitespace for example (why: we might want to use the whitespace as a boundary to extract individual words).

#### Pre-processing data can have varying impact on your outputs and should be carefully considered when thinking of text based analyses. For example there are a number of conventional text cleaning steps typical in many text analyses and some maybe a convention within your domain:

1. Stemming/ Lemmatization
  - Stemming as a function will convert words to their root forms by stripping affixes (in Python NLTK library the porter stemmer is available as well as snowball). Lemmatization preserves some meaning by converting words to their dictionary form to ensure outputs are existing terms.

 > Example: 
  >- Original: There is nothing either good or bad but thinking makes it so.
  >- `Lemmatized: There be nothing either good or bad but think make it so .`
  >- `Stemmed: ['there', 'is', 'noth', 'either', 'good', 'or', 'bad', 'but', 'think', 'make', 'it', 'so', '.']`
2. n-gram creation/inclusion, 
  - n-grams refer to consecutive sequences of words (tokens)
  >- A single word term - unigram `space`
  >- Two words - a bi-gram `space cadet`
  >- Trigram - 3 words `space cadet explorers`
   - The choice of which representation you want to maintain in your corpus will depend on your goals, resulting sparsity and size when you tokenize and a number of considerations. 
3. removal of rare words/combinations 
  - This is a project based consideration, in some workflows having rare words and combinations can act as anomalies and create skews, in other contexts, these may exactly the items one is interested in or would like to maintain.
4. Some trimming down to manageable feature size 
Trimming in this tutorial refers to reduction of corpus by adding thresholds to term frequency at either document or corpus level
  >- document frequency 
  >- corpus frequency
  - feature reduction or dimension reduction is a practice that exist in many data analyses with justification to methodologies or model applications where reduction is needed or clearing of redundant features and size of corpus or dataset is required.

> Like any good wrangling steps in a data analysis workflow, a review of functions on the text itself is important before final application where it might impact interpretability later on. Whilst some steps might reduce dimension, you may also see distortions which might complicate downstream analysis (note machine readable text is different than human readable BUT the metric outputs needs to be placed in context by you). In the sister R version of this tutorial a tool called `preTEXt` is used during pre-processing steps which was designed to aid users in working through these processing steps and to evaluate them before final applications, it can be considered a diagnostic step. In Python the same steps can be applied and evaluated but not through an all encompassing package.


Example of processing data and its outputs is placed below:

## 4b. Text pre-processing considerations

## 4c. Tutorial teaching point
#### Below is a test example case for your review - impact of stemming as a demonstration (not used for the main workflow but as a teaching device):

In [90]:
porter_stemmer = PorterStemmer() #loading up the porter stemmer for word stemming

Test case

In [ ]:
sample_text = cleaned_speech_texts[0][:500] #creating a sample set to demonstrate

original_tokens = re.findall(r"[a-zA-Z']+", sample_text)
stemmed_tokens = [porter_stemmer.stem(token) for token in original_tokens]

token_comparison_df = pd.DataFrame({
    "original_token": original_tokens[:40],
    "stemmed_token": stemmed_tokens[:40]
})

display(token_comparison_df)

Some of these cases look fairly sensible and in some cases such as 'house' to hous' depending on your goal could be an issue. At this stage stop words are still present, nonetheless a word like received is a full entity and when stemmed is stringently formatted to receiv. In some cases this is no issue for small scale projects where interpretation is not hindered and post processing can be used. In cases where you have many forms of terms which can be merged then this may still be useful. Alternatively some researchers may use lemmatizing which is available in NLTK and spaCY, though some extra preparation may be needed before applying (some requisites includes Parts Of Speech -POS tagging).

## 5. Continuing with data handling before LDA model application

Running conventional text handling methods and reviewing outputs on sample text can provide some insights on a way forward. In this tutorial the diagnostic tests were ran in an R based package and the outputs suggested that
- Stemming
- n-gram inclusion
- stop word removal 
- Punctuation removal
were viable supported steps for processing the data.

> In the python tutorial we also explore each of these steps, however stemming is kept as **optional** in the code blocks and is highlighted. If you would like to include it, simply remove the # from the code line, this should be completed consistently and token representations will be updated to include this. 

## 5a. Exploring tokenisation

The next step is tokenisation, which means splitting the text into individual units called tokens. In many cases, these tokens are single words. Tokenisation helps to make machine readable elements which can be used to form representations of the corpus to use for model application later in the workflow.

Tokenisation is necessary for topic modelling because LDA works with counts of terms across documents. Before a document-term matrix can be constructed, the text must therefore be broken into analysable units.

In the Python workflow below, tokenisation is explored explicitly through custom functions so that the effect of different preprocessing choices can be inspected directly.

In [92]:
def tokenise_with_punctuation(text):
    # split text by whitespace while retaining punctuation within tokens
    return text.split()

def tokenise_no_punctuation(text):
    # extract only word-like tokens made of letters and apostrophes
    return re.findall(r"[a-zA-Z']+", text)

def tokenise_no_punctuation_no_stopwords(text):
    # remove punctuation first, then remove common English stopwords
    tokens = tokenise_no_punctuation(text)
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS and len(token) > 2]

# Optional stemming version here for comparison, but not used by default
# def tokenise_stemmed(text):
#     tokens = tokenise_no_punctuation_no_stopwords(text)
#     return [porter_stemmer.stem(token) for token in tokens]

Take a look at the pre-processing steps so far

In [ ]:
print("Tokens with punctuation retained:")
print(textwrap.fill(" ".join(tokenise_with_punctuation(cleaned_speech_texts[0])[:30]), width=100))

print("\nTokens without punctuation:")
print(textwrap.fill(" ".join(tokenise_no_punctuation(cleaned_speech_texts[0])[:30]), width=100))

print("\nTokens without punctuation and stopwords:")
print(textwrap.fill(" ".join(tokenise_no_punctuation_no_stopwords(cleaned_speech_texts[0])[:30]), width=100))

# stemming version to inspect
# print("\nStemmed tokens:")
# print(textwrap.fill(" ".join(tokenise_stemmed(cleaned_speech_texts[0])[:30]), width=100))

Quick note -> punctuation removal and stop word removal were useful here and we can proceed with these and move forward

## 5b. Applied tokenisation & Document Term Matrix

> We would typically have a list of preprocessing agreed upon and apply this directly but in this tutorial, we explore them and iteratively make decisions before including them in our final Document Term Matrix

Above we explored tokens with punctuation, tokens without punctuation and then an iterative tokens with punctuation AND stopwords removed. Note below, we will move forward with tokens without punctuation and stop words. Optionally stemmed tokens which would inhabit the latter processing too.

What is a Document Term Matrix? -> 1 speech is 1 document

From the data inspection earlier we know we have 60 speeches in rows
- rows-> documents (speech text is contained within)
- columns->terms (tokenised)
- values->counts of terms in each document
so we would have 60 documents with 1000s of terms, per document we may have a number of topics and are interested in looking at the proportional distributions of those topics within the documents through terms (their features).

#### Count vectoriser from scikit learn-> to tokenise & form DTM

In [ ]:
# Main workflow: remove punctuation and stopwords, but do not apply stemming
count_vectoriser = CountVectorizer(
    tokenizer=tokenise_no_punctuation_no_stopwords, # THIS IS THE TOKENIZER WE MOVE FORWARD WITH
    token_pattern=None,
    lowercase=False,   # text has already been lowercased during basic cleaning
    ngram_range=(1, 1),
    min_df=1
)

# Optional **alternative**: uncomment this version if you want stemming included
# count_vectoriser = CountVectorizer(
#     tokenizer=tokenise_stemmed, # THIS LINE IS USED TO DEFINE THE TOKENIZER WE USE.
#     token_pattern=None,
#     lowercase=False,
#     ngram_range=(1, 1),
#     min_df=1
# )

###### IF YOU USE THE STEMMED TOKENISER, THEN # OUT THE MAIN TOKENISER ABOVE.#######

document_term_matrix = count_vectoriser.fit_transform(cleaned_speech_texts)

print("DTM shape:", document_term_matrix.shape)
print("No. of features:", len(count_vectoriser.get_feature_names_out()))

print("\nFirst 20:")
print(count_vectoriser.get_feature_names_out()[:20])

# **CountVectorizer(stop_words="english")** can remove stopwords automatically, 
#this tutorial uses an explicit tokenisation function so that punctuation handling, stopword removal, 
# and optional stemming can be demonstrated

> The outputs show the first 20 features in alphabetical order for a brief review. Note here for the main steps where we have not applied stemming, both abandoned and abandon are included so far.

#### What can we see from the terms and their frequency so far?
Ranked frequency order for inspection 

In [ ]:
term_frequencies = document_term_matrix.sum(axis=0).A1
feature_names = count_vectoriser.get_feature_names_out()

term_frequency_df = pd.DataFrame({
    "term": feature_names,
    "frequency": term_frequencies
}).sort_values("frequency", ascending=False)

display(term_frequency_df.head(20))

## 5c. Sparsity checks

Sparsity measures how many cells in the document-term matrix are at zero. Text data is usually sparse because each document contains only a small fraction of the total vocabulary (in this case the vocabulary size was quite large 1000s of terms), when we have rare terms in the mix, terms which occur infrequently across the matrix, we have more empty cells. Why would this matter? Each speech may have a small portion of vocabulary when the document term matrix is constructed and assessed. This is not surprising for text analyses but the sparsity indexes can act as a guiding metric when handling the data. The value can also be conformed to a %. Sparsity will matter again as we go along

> Important note, so far we have used ngram (1,1) which only single word terms making this a unigram level review. We will examine and compare the value of unigram and bigram further in the workflow.

In [ ]:
unigram_matrix_sparsity = 1 - (
    document_term_matrix.count_nonzero()
    / (document_term_matrix.shape[0] * document_term_matrix.shape[1])
)

print("Unigram document-term matrix sparsity:", round(unigram_matrix_sparsity, 4))

Plotting terms

In [ ]:
unigram_top_terms_df = term_frequency_df.head(15).sort_values("frequency", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(unigram_top_terms_df["term"], unigram_top_terms_df["frequency"])
plt.xlabel("Frequency")
plt.ylabel("Unigram term")
plt.title("Top 15 unigram terms in the corpus")
plt.show()

We are working through options iteratively to make decisions to handling the data and reviewing before application. So far we have completed basic cleaning, DTM formation with removal of punctuation and stop words, (no stemming in the formal workflow) but was added as an option for post class review and exploration and unigram inclusion.

## **IMPORTANT!**

This is an adaptable workflow and you can choose to use stemming as mentioned above with some simple hashing out of the # in code blocks. If you encounter countvectorizer in future blocks


> In later parts of the notebook, you will encounter `CountVectorizer(...)` blocks such as:
>
> ```python
> count_vectoriser_unigrams_bigrams = CountVectorizer(
>     tokenizer=tokenise_no_punctuation_no_stopwords,
>     token_pattern=None,
>     lowercase=False,   
>     ngram_range=(1, 2),
>     min_df=1
> )
> ```
>
> If you selected stemming earlier, remember to replace:
>
> 
> `tokenizer=tokenise_no_punctuation_no_stopwords`
> 
> with:
> 
> `tokenizer=tokenise_stemmed`


## 6. Iterative building - considering further representations

## *Decision point*

## 6a. Uni-grams and bi-grams

Here we adding a further representation of bi-grams and will reach a decision point whether we move forward with just unigrams or both. If this was within the remit of your work you would have added it in your earlier `CountVectorizer` step via the line specifying `ngram` as seen below. 

Lets review how it works and what its application might look like:

In [ ]:
# A second CountVectoriser that includes both unigrams and bigrams.
# This allows the document-term matrix to capture single words AND short phrases like 'house representatives' or 'abandon hope' and such for example.

count_vectoriser_unigrams_bigrams = CountVectorizer(
    tokenizer=tokenise_no_punctuation_no_stopwords, #change this to stemmed tokenizer if you want to follow the alternate workflow
    token_pattern=None,
    lowercase=False,  
    ngram_range=(1, 2), # this is where the bi-gram dimension is incorporated, earlier it was ngram_range=(1,1), for bigrams only use (2,2)
    min_df=1
)

document_term_matrix_unigrams_bigrams = count_vectoriser_unigrams_bigrams.fit_transform(cleaned_speech_texts)

print("DTM shape (unigrams + bigrams):", document_term_matrix_unigrams_bigrams.shape)
print("No. of features (unigrams + bigrams):", len(count_vectoriser_unigrams_bigrams.get_feature_names_out()))

print("\nFirst 30 features:")
print(count_vectoriser_unigrams_bigrams.get_feature_names_out()[:30])
# for those who include stemming provided that all stemming related code lines were ran that should be embedded in the DTM we are working from

The output shows the inclusion of rare phrase combination but richer context - adding so much to the feature space -> take a look at the dimension shape now, it is HUGE. This point is a user decision and depends on your research framework. It is important to note however, that expanded features is not always a plus, as mentioned earlier it can add volatility to the distributions however in some cases this type of feature breakdown could be used to understand the scope of the corpus and what can be extracted.

### Do the top term frequencies look majorly different?

In [ ]:
term_frequencies_unigrams_bigrams = document_term_matrix_unigrams_bigrams.sum(axis=0).A1
feature_names_unigrams_bigrams = count_vectoriser_unigrams_bigrams.get_feature_names_out()

term_frequency_unigrams_bigrams_df = pd.DataFrame({
    "term": feature_names_unigrams_bigrams,
    "frequency": term_frequencies_unigrams_bigrams
}).sort_values("frequency", ascending=False)

display(term_frequency_unigrams_bigrams_df.head(30))

#### At just the bi-gram level what kinds of features are present?

In [ ]:
bigram_only_frequency_df = term_frequency_unigrams_bigrams_df[
    term_frequency_unigrams_bigrams_df["term"].str.contains(" ")
]

print("Top bigram terms:")
display(bigram_only_frequency_df.head(20))

The first column is simply an index. Looking at the frequency of the bigrams we added to the representation they look largely sensible, and depending on analysis goals and research question this form may be insightful for model application. In some cases the uni-gram maybe more concise for term level therefore we can compare via sparsity checks and a close look before the next decision point.

## 6b. Sparsity checks

We see sparsity levels shift again, adding bigrams which are rare terms, and show up infrequently across the DTM, expand the space where many zeros will exist. See above "years ago"

In [ ]:
unigram_matrix_sparsity = 1 - (
    document_term_matrix.count_nonzero()
    / (document_term_matrix.shape[0] * document_term_matrix.shape[1])
)

unigrams_bigrams_matrix_sparsity = 1 - (
    document_term_matrix_unigrams_bigrams.count_nonzero()
    / (
        document_term_matrix_unigrams_bigrams.shape[0]
        * document_term_matrix_unigrams_bigrams.shape[1]
    )
)

print("Unigram matrix sparsity:", round(unigram_matrix_sparsity, 4))
print("Unigram + bigram matrix sparsity:", round(unigrams_bigrams_matrix_sparsity, 4))

## *Decision point* - ngrams

In the earlier blocks we aimed to compare candidate document-term representations, we already moved forward with a set of pre-processing steps and now we want to decide on unigram only, bigram only or both unigram and bigram representation. Outputs showed that the most frequent terms were still largely held by unigram representations, yet some of the bigrams seemed richer for context. Since this is an exploratory analysis and teaching tool, the direction can be fluid. In research projects there may be a defined reasons for why the sparsity comparisons should point your direction.

Unigrams offer:
- fewer features
- less sparsity
- simpler interpretation of the DTM
- A good exploratory space

Combined unigrams + bigrams:

- may maintain meaningful phrases
- a richer representation of the speeches

In this case this representation follows well with the sister tutorial in R and we proceed with the combined features.


## 7. Feature handling

## *Decision point* - expanded feature space and thresholding

We have expanded our feature space somewhat and can prune some features down to more meaningful sets, where meaningful is more so a data driven threshold than qualitative. Some thresholding or informally trimming would be a sensible step. A key point to note here is that this type of pruning can happen much earlier in the workflow when pre-processing the data. As a demonstrative case, the tutorial hopes to show how these cases might be handled in your given research setting

## 7a. OPTION 1 - Document frequency trimming

In [ ]:
# Option 1: trim by document frequency.
# Here, min_df=3 keeps only terms that appear in at least 3 documents.

document_frequency_count_vectoriser = CountVectorizer(
    tokenizer=tokenise_no_punctuation_no_stopwords,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=3 #this is the key line
)

# Optional alternative if stemming is part of the workflow:
# document_frequency_count_vectoriser = CountVectorizer(
#     tokenizer=tokenise_stemmed,
#     token_pattern=None,
#     lowercase=False,
#     ngram_range=(1, 2),
#     min_df=3
# )

document_frequency_document_term_matrix = (
    document_frequency_count_vectoriser.fit_transform(cleaned_speech_texts)
)

document_frequency_trimmed_feature_names = (
    document_frequency_count_vectoriser.get_feature_names_out()
)

print(
    "DTM trimmed matrix shape:",
    document_frequency_document_term_matrix.shape
)
print(
    "Number of features after document-frequency trimming:",
    len(document_frequency_trimmed_feature_names)
)

checking what happened with the trimming (feature reduction)

In [ ]:
all_feature_names_unigrams_bigrams = count_vectoriser_unigrams_bigrams.get_feature_names_out()
all_document_frequencies_unigrams_bigrams = (
    (document_term_matrix_unigrams_bigrams > 0).sum(axis=0).A1
)

all_terms_df = pd.DataFrame({
    "term": all_feature_names_unigrams_bigrams,
    "document_frequency_before_trimming": all_document_frequencies_unigrams_bigrams
})

document_frequency_removed_terms_df = all_terms_df[
    ~all_terms_df["term"].isin(document_frequency_trimmed_feature_names)
].sort_values(["document_frequency_before_trimming", "term"]).reset_index(drop=True)

display(document_frequency_removed_terms_df.head(50))

We can save these for a closer look

In [105]:
document_frequency_removed_terms_df.to_csv(
    outputs_dir / "7a_document_frequency_removed_terms.csv",
    index=False
)
#check the output folder for the file

Sparsity is improved.

In [ ]:
document_frequency_matrix_sparsity = 1 - (
    document_frequency_document_term_matrix.count_nonzero()
    / (
        document_frequency_document_term_matrix.shape[0]
        * document_frequency_document_term_matrix.shape[1]
    )
)

print(
    "Document-frequency trimmed matrix sparsity:",
    round(document_frequency_matrix_sparsity, 4)
)

Comparing the effects of feature reduction using document frequency on the terms

In [107]:
document_frequency_term_frequencies = document_frequency_document_term_matrix.sum(axis=0).A1

document_frequency_term_frequency_df = pd.DataFrame({
    "term": document_frequency_trimmed_feature_names,
    "frequency": document_frequency_term_frequencies
}).sort_values("frequency", ascending=False).reset_index(drop=True)
#sorting

In [ ]:
print("Most frequent retained terms after document-frequency trimming:")
display(document_frequency_term_frequency_df.head(20))

print("Least frequent retained terms after document-frequency trimming:")
display(
    document_frequency_term_frequency_df.sort_values("frequency", ascending=True)
    .head(20)
    .reset_index(drop=True)
)
#displaying

## 7b. Option 2 for trimming - corpus wide

In the sister workflow for the workshop in R, a corpus wide threshold was used for trimming down. Below the option is applied and reviewed for inspection. Finally a comparison of how both option 1 at document level and option 2 at corpus level can have varying results in the sparsity index is shown

In [ ]:
# Option 2 is trim by total corpus frequency
# Keep only terms that occur at least 3 times across the whole corpus.

full_feature_names_unigrams_bigrams = (
    count_vectoriser_unigrams_bigrams.get_feature_names_out()
)
full_term_frequencies_unigrams_bigrams = (
    document_term_matrix_unigrams_bigrams.sum(axis=0).A1
)

corpus_frequency_mask = full_term_frequencies_unigrams_bigrams >= 3

corpus_frequency_trimmed_feature_names = (
    full_feature_names_unigrams_bigrams[corpus_frequency_mask]
)
corpus_frequency_trimmed_document_term_matrix = (
    document_term_matrix_unigrams_bigrams[:, corpus_frequency_mask]
)

print(
    "Corpus-frequency trimmed matrix shape:",
    corpus_frequency_trimmed_document_term_matrix.shape
)
print(
    "Number of features after corpus-frequency trimming:",
    len(corpus_frequency_trimmed_feature_names)
)

Sparsity check

In [110]:
corpus_frequency_matrix_sparsity = 1 - (
    corpus_frequency_trimmed_document_term_matrix.count_nonzero()
    / (
        corpus_frequency_trimmed_document_term_matrix.shape[0]
        * corpus_frequency_trimmed_document_term_matrix.shape[1]
    )
)

print(
    "Corpus-frequency trimmed matrix sparsity:",
    round(corpus_frequency_matrix_sparsity, 4)
)

Corpus-frequency trimmed matrix sparsity: 0.8804


In [111]:
corpus_frequency_term_frequencies = (
    corpus_frequency_trimmed_document_term_matrix.sum(axis=0).A1
)

corpus_frequency_term_frequency_df = pd.DataFrame({
    "term": corpus_frequency_trimmed_feature_names,
    "frequency": corpus_frequency_term_frequencies
}).sort_values("frequency", ascending=False).reset_index(drop=True)
#sorting

In [ ]:
print("Most frequent retained terms after corpus-frequency trimming:")
display(corpus_frequency_term_frequency_df.head(20))

print("Least frequent retained terms after corpus-frequency trimming:")
display(
    corpus_frequency_term_frequency_df.sort_values("frequency", ascending=True)
    .head(20)
    .reset_index(drop=True)
)
#displaying

## 7c. Removing features

In [ ]:
all_terms_corpus_frequency_df = pd.DataFrame({
    "term": full_feature_names_unigrams_bigrams,
    "corpus_frequency_before_trimming": full_term_frequencies_unigrams_bigrams
})

corpus_frequency_removed_terms_df = all_terms_corpus_frequency_df[
    ~all_terms_corpus_frequency_df["term"].isin(corpus_frequency_trimmed_feature_names)
].sort_values(["corpus_frequency_before_trimming", "term"]).reset_index(drop=True)

display(corpus_frequency_removed_terms_df.head(50))

#### Saving for inspection and reference (an example)

In [114]:
corpus_frequency_removed_terms_df.to_csv(
    outputs_dir / "7c_corpus_frequency_removed_terms.csv",
    index=False
)

# see the outputs folder

## 7d. Compare the two options for trimming features

In [ ]:
comparison_trimming_df = pd.DataFrame({
    "trimming_method": [
        "document_frequency_min_df_3",
        "corpus_frequency_min_3"
    ],
    "n_documents": [
        document_frequency_document_term_matrix.shape[0],
        corpus_frequency_trimmed_document_term_matrix.shape[0]
    ],
    "n_features": [
        document_frequency_document_term_matrix.shape[1],
        corpus_frequency_trimmed_document_term_matrix.shape[1]
    ],
    "sparsity": [
        document_frequency_matrix_sparsity,
        corpus_frequency_matrix_sparsity
    ]
})

display(comparison_trimming_df)

Both conservative trimming approaches substantially reduced the feature space and produced document-term matrices with similar sparsity (though document level thresholding was more stringent, option 1). The corpus-frequency approach retained slightly more terms and more closely matches the logic of the original R workflow, which removed rare terms based on total frequency across the corpus. For this reason, the corpus-frequency trimmed matrix is selected as the final input for topic modelling.

The two trimming methods produce similar results, but the corpus-frequency approach retains slightly more terms and is closer to the logic of the sister R workflow for this tutorial. For this reason, the corpus-frequency trimmed matrix is selected as the final modelling input.

## 8. Final choices for Latent Dirichlet Allocation model inputs & application

In [ ]:
final_trimmed_feature_names = corpus_frequency_trimmed_feature_names
final_document_term_matrix = corpus_frequency_trimmed_document_term_matrix
final_trimming_method = "corpus_frequency_min_3"
# we set all of the final selected inputs we will use for LDA modelling and list them here.
print("option 2 trimming method:", final_trimming_method)
print("Final matrix shape:", final_document_term_matrix.shape) # 60 documents
print("Final no. of features:", final_document_term_matrix.shape[1]) 

Now that core decisions for data handling have been selected we can select number of topics and for the Latent Dirichlet Allocation model. We still have some decisions to make requiring both domain convention , review of outputs so far and linking back to the data. The main decision point here is the number of topics (K) for selection when fitting the model and evaluating the best direction for the k and the topics

## 8a. Choosing topic numbers (i.e what is our K)?

Goal for this sub-section is to find the best k for the LDA model(with k being the number of topics in our DTM):
We can use log-likelihood metric to guide us and we will start with fit an LDA model for each k in a set range, score the fitted model
assess the log-likelihood (which tells us about the fit of the model), however we must still discern between the outputs since topics ought to be interpretable. The idea is well distinguished topics that can be thematically guided and provide insights with your research.
> Applying LDA model to infinite k number iterations until you land on a set which best suits a narrative is NOT the goal. Depending on the corpus, representations, data handling earlier, and a well defined question the outputs are an exploratory guide which still require user insights for qualitative additions, and placing topics into context. Having a range to assess is useful in this case with added metrics which you will examine in this section.

In [ ]:
# Compare LDA models across a range of topic numbers using log-likelihood.
# This comparison is run on the final corpus-frequency-trimmed unigram + bigram
# document-term matrix built from cleaned text with punctuation removed and stopwords removed.

k_grid_values = list(range(2, 13))
k_grid_lda_models = {}
k_grid_log_likelihood_values = []

for k in k_grid_values:
    k_grid_lda_topic_model = LatentDirichletAllocation(
        n_components=k,
        random_state=1234,      # fixed seed for reproducibility
        learning_method="batch",
        max_iter=20
    )

    k_grid_lda_topic_model.fit(final_document_term_matrix)

    k_grid_lda_models[k] = k_grid_lda_topic_model
    k_grid_log_likelihood_values.append(
        k_grid_lda_topic_model.score(final_document_term_matrix)
    )

topic_selection_df = pd.DataFrame({
    "k": k_grid_values,
    "log_likelihood": k_grid_log_likelihood_values
})

display(topic_selection_df)

## 8b. Log Likelihood by K

Plot for viewing - hover over data points.

In [ ]:
topic_selection_plot = px.line(
    topic_selection_df,
    x="k",
    y="log_likelihood",
    markers=True,
    title="Log-likelihood across candidate numbers of topics",
    hover_data={
        "k": True,
        "log_likelihood": ":.3f"
    }
)

topic_selection_plot.update_layout(
    xaxis_title="Number of topics (k)",
    yaxis_title="Log-likelihood"
)

topic_selection_plot.update_xaxes(dtick=1)

topic_selection_plot.show()

Save the html file in output folder to view in a browser (after downloading)

In [119]:
topic_selection_plot.write_html(
    outputs_dir / "8b_log_likelihood_by_k.html"
)

The highly negative values suggest some skewing. K of 2 which is topic number of 2 is the better suited option here. However, a number of reasons can explain why the range dips as the K is increased. Sparsity, rare terms, model features, LDA implementation type, and many other factors. We can still assess qualitatively to decide and add coherence score.

> In some packages you will also note a perplexity score, it is difficult to interpret and therefore was not used in this workflow.

## 8c. Coherence
In this notebook, log-likelihood is used as a fit-based metric and coherence is added as a second, interpretability-oriented evaluation metric. These are then considered alongside qualitative inspection of the topic terms before selecting a final value of k which inspecting with human insight lenses for the topics.

> Note: we borrow gensim coherence score functions here. Read more [here](https://radimrehurek.com/gensim/models/coherencemodel.html)

In [120]:
def get_top_terms_per_topic(model, feature_names, n_top_terms=10):
    topic_rows = []

    for topic_number, topic_weights in enumerate(model.components_, start=1):
        top_term_indices = topic_weights.argsort()[::-1][:n_top_terms]

        for rank, term_index in enumerate(top_term_indices, start=1):
            topic_rows.append({
                "topic": topic_number,
                "rank_within_topic": rank,
                "term": feature_names[term_index],
                "weight": topic_weights[term_index]
            })

    return pd.DataFrame(topic_rows)

prepping for coherence model checks

In [121]:
coherence_tokenised_texts = [
    tokenise_no_punctuation_no_stopwords(text) ###change to stemmed tokenizer is you applied it earlier 'tokenise_stemmed'
    for text in cleaned_speech_texts
]

coherence_dictionary = Dictionary(coherence_tokenised_texts)

Applying coherence metric for evaluation

In [ ]:
k_grid_coherence_values = []

for k in k_grid_values:
    k_grid_top_terms_df = get_top_terms_per_topic(
        k_grid_lda_models[k],
        final_trimmed_feature_names,
        n_top_terms=10
    )

    topic_term_lists = []
    for topic_number in sorted(k_grid_top_terms_df["topic"].unique()):
        topic_terms = (
            k_grid_top_terms_df[k_grid_top_terms_df["topic"] == topic_number]
            .sort_values("rank_within_topic")["term"]
            .tolist()
        )
        topic_term_lists.append(topic_terms)

    coherence_model = CoherenceModel(
        topics=topic_term_lists,
        texts=coherence_tokenised_texts,
        dictionary=coherence_dictionary,
        coherence="c_v"
    )

    k_grid_coherence_values.append(coherence_model.get_coherence())

k_grid_model_summary_df = pd.DataFrame({
    "k": k_grid_values,
    "log_likelihood": [
        topic_selection_df.loc[topic_selection_df["k"] == k, "log_likelihood"].iloc[0]
        for k in k_grid_values
    ],
    "coherence_c_v": k_grid_coherence_values
})

display(k_grid_model_summary_df) #may take time to compile

In [ ]:
coherence_plot = px.line(
    k_grid_model_summary_df,
    x="k",
    y="coherence_c_v",
    markers=True,
    title="Coherence score across candidate numbers of topics",
    hover_data={"k": True, "coherence_c_v": ":.4f"}
)

coherence_plot.update_layout(
    xaxis_title="Number of topics (k)",
    yaxis_title="Coherence (c_v)"
)

coherence_plot.update_xaxes(dtick=1)

coherence_plot.write_html(
    outputs_dir / "8c_coherence_score_by_k.html"
)
coherence_plot.show()

Gensim’s CoherenceModel can evaluate topic models from other libraries as long as the topic-word output is provided in the required format. In this workflow, the top words from each scikit-learn topic are extracted and passed to CoherenceModel together with the tokenised texts and a matching dictionary.

Now that we have a coherence score too, we can evaluate both log likelihood (model fit based) and coherence in tandem. Interestingly the two metrics point to different number of K for our choices, for instance K=2 topics is optimal by the log likelihood measure, however coherence suggests K=9 and k=11 are the highest and could be taken forward. The additional metric of coherence offers some further options.

## 8d. Specified topic numbers for closer inspection (qualitative look)

The candidate models below are inspected using their top-weighted terms. Within a given model, larger weights indicate that a term is more strongly associated with a topic. The weights exist for ranking terms within topics and do not serve as comparators between different K values since each model has a different topic structure.

In [124]:
inspection_k_values = [2, 3, 4, 5, 6, 7]
inspection_lda_models = {k: k_grid_lda_models[k] for k in inspection_k_values}

In [ ]:
inspection_model_summary_df = topic_selection_df[
    topic_selection_df["k"].isin(inspection_k_values)
].copy()

inspection_model_summary_df["fit_rank"] = (
    inspection_model_summary_df["log_likelihood"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

inspection_model_summary_df = inspection_model_summary_df.sort_values("k").reset_index(drop=True)

display(inspection_model_summary_df)

## 8e. K inspections 2, 3, 4, 5, 6, 7, 8, 9, 11

In [ ]:
for k in inspection_k_values:
    print(f"\nTop terms for k = {k}")
    inspection_top_terms_df = get_top_terms_per_topic(
        inspection_lda_models[k],
        final_trimmed_feature_names,
        n_top_terms=10
    )
    display(inspection_top_terms_df)

#### Grouping topics for each K to asses the ranking of terms

In [ ]:
for k in inspection_k_values:
    inspection_top_terms_df = get_top_terms_per_topic(
        inspection_lda_models[k],
        final_trimmed_feature_names,
        n_top_terms=10
    )

    print(f"\nTop terms grouped by topic for k = {k}")
    display(
        inspection_top_terms_df.pivot(
            index="rank_within_topic",
            columns="topic",
            values="term"
        )
    )

Save file for inspection and review

In [128]:
inspection_topics_weights_excel_path = outputs_dir / "8d_inspection_topic_terms_with_weights_by_k.xlsx"

with pd.ExcelWriter(inspection_topics_weights_excel_path, engine="openpyxl") as writer:
    for k in inspection_k_values:
        inspection_top_terms_df = get_top_terms_per_topic(
            inspection_lda_models[k],
            final_trimmed_feature_names,
            n_top_terms=10
        )

        inspection_top_terms_df = inspection_top_terms_df[
            ["topic", "rank_within_topic", "term", "weight"]
        ]

        inspection_top_terms_df.to_excel(writer, sheet_name=f"k_{k}", index=False)

By eye and qualitative review many of the topics as K increases do not become **very** thematically distinguishable but some features do emerge. One of the key issues however is where terms are shared too often between topics and their proportion is overrepresented in the topics. The log likelihood output shows ideally a K of two, but for instance citizens occurring as a high weighted term in two topics in context could not be straying too far, since the reference could be to internal, or external citizens of other nations and is not an atypical term that would be present across topics. One caveat is whether log likelihood works best here as the evaluation metric considering the scikit learn LDA implementation compared to R topic models which uses differing methods for topic inferences. One thing we considered in addition was a 'coherence score' which can provide a dual guide.

## *Summary so far...*

So far we have iterated through data preparation steps, looked at the varying approaches and at each point made a selection which was then used to update the representations we wanted to use for LDA. For instance we had a look at unigram (n-gram) representation as well as bi-grams, and proceeded forward with the dual representation. The same was shown for assessing punctuation removal to demonstrate the impact in the tutorial. However, in a typical workflow standard processing and data handling might be decided earlier and ahead of time to fit your research remit and would be applied in a more linear format than the demonstrative flow we have used. In the case of choosing K, an assessment through log likelihood, coherence scores is a sensible approach with added inspection, and domain convention based on the corpus size too. In some cases you might return to earlier decision points and consider further refinement, for instance the trimming options we showed could have been applied in much earlier stages for reducing features. Sparsity can be assessed when DTM are formed and is the guiding principle for some of the feature related decisions.

## 9. Choosing the final K choice and applied LDA 

A topic is a bag of many words with different weights. Sometimes probabilities are easier to understand  than weights since they are restricted between 0-1. In some outputs the probability value is included for each weight per term. The probability tells us how much of that bag belongs to each word. The most informative words are the ones with the biggest shares of the total. Though, the values you encounter might be very small.

Let's say we proceed with k of 4 as a mid ground exploratory analysis choice:
> higher values such as k = 9 showed stronger coherence, choosing this could result in  models becoming more fragmented and harder to interpret. As in between metric and an inspection of topics, 4 is a good option for an exploratory analysis. For the purposes of this tutorial, k = 4 provides a more balanced and readable solution that still allows meaningful topic differentiation

In [ ]:
final_k = 4

final_lda_topic_model = LatentDirichletAllocation( #we do this here to store this as the final model application iteration
    n_components=final_k,
    random_state=1234,
    learning_method="batch",
    max_iter=20
)

final_lda_topic_model.fit(final_document_term_matrix)

## Spanning relevance per topic (and related terms)
Use the pyLDAviz visualisation to span, hover and explore the 4 topics, try and toggle the relevance metric and see how items shift and change

In [ ]:
topic_term_dists = final_lda_topic_model.components_ / final_lda_topic_model.components_.sum(axis=1)[:, None]
doc_topic_dists = final_lda_topic_model.transform(final_document_term_matrix)
doc_lengths = final_document_term_matrix.sum(axis=1).A1
vocab = list(final_trimmed_feature_names)
term_frequency = final_document_term_matrix.sum(axis=0).A1

pyldavis_prepared = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dists,
    doc_lengths=doc_lengths,
    vocab=vocab,
    term_frequency=term_frequency,
    sort_topics=False
)

pyLDAvis.save_html(
    pyldavis_prepared,
    str(outputs_dir / "9_final_lda_pyldavis.html")
)

pyLDAvis.display(pyldavis_prepared) #may take time to load

The file has saved in the output directory, so it can be downloaded and viewed in a browser (with a double click) for clearer viewing.

The visual has two linked components, an intertopic distance map and a term relevance bar chart
In the map on the left
- Each circle is a topic.
- Circle size reflects the topic’s overall prevalence in the corpus.
- Distance between circles reflects how different the topics are from one another in their term distributions.
- Topics that are far apart are more distinct.
- Topics that overlap or sit close together may share many similar words or themes.

When you hover over the circles the bar chart will adjust
The bar chart compares
- the overall frequency of a term in the corpus
- the frequency or relevance of that term within the selected topic

The relevance slider (lamda)
- when lamda is lower you would see terms that are more distinctive to the topic
- when lamda is higher as in closer 1, then you see terms most probably in that topic

Bonus task: check the topic sizes via the circles, click through them sequentially, keep an eye on relevant terms on the right, play around with the slider.

Topics are model representations so these views help to explore the fitted model.

## 9a. Inspecting the top terms (up to 10)

In [ ]:
def get_topic_term_summary(model, feature_names, n_top_terms=10):
    topic_rows = []

    for topic_number, topic_weights in enumerate(model.components_, start=1):
        topic_probabilities = topic_weights / topic_weights.sum()
        top_term_indices = topic_weights.argsort()[::-1][:n_top_terms]

        for rank, term_index in enumerate(top_term_indices, start=1):
            topic_rows.append({
                "topic": topic_number,
                "rank_within_topic": rank,
                "term": feature_names[term_index],
                "weight": topic_weights[term_index],
                "probability": topic_probabilities[term_index]
            })

    return pd.DataFrame(topic_rows)

final_topic_term_summary_df = get_topic_term_summary(
    final_lda_topic_model,
    final_trimmed_feature_names,
    n_top_terms=10
).reset_index(drop=True)

display(final_topic_term_summary_df)

Looking at k=4 topic outputs (rankings side by side)

In [ ]:
display(
    final_topic_term_summary_df.pivot(
        index="rank_within_topic",
        columns="topic",
        values="term"
    )
)

## 9b. Final outputs

In [ ]:
final_topic_term_plot_df = final_topic_term_summary_df.copy().sort_values(
    ["topic", "rank_within_topic"],
    ascending=[True, True]
).reset_index(drop=True)

final_topic_term_plot_df["topic_label"] = (
    "Topic " + final_topic_term_plot_df["topic"].astype(str)
)

final_topic_term_plot_df["term_label"] = final_topic_term_plot_df["term"]
final_topic_term_plot_df["rank_label"] = (
    "Rank " + final_topic_term_plot_df["rank_within_topic"].astype(str)
)

display(final_topic_term_plot_df.head())

Saving the file for reference

In [134]:
final_topic_term_plot_df.to_csv(
    outputs_dir / "9b_final_topic_term_summary.csv",
    index=False
)

# 10. Interpretations and researcher contextualisations


## 10a. Visualising k=4 (topics)

In [ ]:
max_topic_probability = final_topic_term_plot_df["probability"].max()
x_axis_upper_limit = max_topic_probability * 1.1

final_topic_terms_plot = px.bar(
    final_topic_term_plot_df,
    x="probability",
    y="term_label",
    color="topic_label",
    facet_col="topic_label",
    facet_col_wrap=2,
    orientation="h",
    hover_data={
        "topic": True,
        "topic_label": False,
        "term_label": False,
        "term": True,
        "rank_within_topic": True,
        "weight": ":.3f",
        "probability": ":.4f"
    },
    title="Top terms in each topic",
    labels={
        "probability": "Topic-term probability",
        "term_label": "Term",
        "topic_label": "Topic"
    }
)

final_topic_terms_plot.update_yaxes(
    matches=None,
    categoryorder="total ascending",
    automargin=True
)

final_topic_terms_plot.update_xaxes(
    range=[0, x_axis_upper_limit],
    tickformat=".3f"
)

# y-axis labels per facet
final_topic_terms_plot.for_each_yaxis(
    lambda axis: axis.update(showticklabels=True)
)

final_topic_terms_plot.update_layout(
    showlegend=False,
    height=950,
    margin=dict(t=90, l=120, r=40, b=50),
    font=dict(size=12)
)

final_topic_terms_plot.for_each_annotation(
    lambda a: a.update(
        text=a.text.replace("topic_label=", ""),
        font=dict(size=13)
    )
)

final_topic_terms_plot.show()

The plot is saved in the outputs folder as a html file and can be downloaded and viewed in a browser with hover functions

In [136]:
plot_html_path = outputs_dir / "10a_final_topic_terms_plot.html"
final_topic_terms_plot.write_html(plot_html_path)

## Lets dive into what these topics might or can tell us about our data

 The distinguishing between topics depends on goals, earlier steps in the workflow and a researchers qualitative insights. Here the k=4 was a reasonable point for K choice, however some theme assignment for grouping would help. In some cases terms were shared between topics such as 'government' and 'people' but there were some which were a little more distinguished between the topics that can be used to group these under banners of their own. Increasing the k did not show further differentiation but could be more meaningful with a rsearcher lens for example if the research goal was type of language used for thematic grouping and this assessment is user based beyond the log-likelihood metrics

In clustering analyses which are largely unsupervised, a similar phenomenon can occur where latent patterns captured by a model may not match exact grouping expectations, often times due to multicollinearity, shared features and underlying patterns which make distinct boundaries less clear.

## Mapping speeches to topics

After fitting the final LDA model, we can preview the estimate topic mixture for each speech.

The values returned here are **document-topic probabilities**. For each speech, they show how much of the speech is associated with each topic. These probabilities range from **0 to 1** and, across all topics for a given speech, they sum to **1**.

- values closer to **1** indicate stronger association with that topic
- values closer to **0** indicate weaker association with that topic
- many speeches are a **mixture** of topics rather than belonging entirely to one topic

For example:

- a dominant topic probability of **0.80** is how 'dominance' is categorised for a topic in a given speech
- a value around **0.40–0.50** might point to a more mixed speech


## 10b. Giving the topics some thematic labels (researcher contextualisation)/qualitative framing)

Setting up dataframes to map topics to the data elements for context
> Note this is an exploratory approach for tracing topics to data. The idea is to link each speech to its estimated topic mixture under the fitted LDA model. The topic probabilities are inferred from the term patterns in the speech and should be interpreted as model-based estimates rather than exact measurements. The dominant topic is simply the topic with the highest estimated probability for that speech (highest value in a row), and the dominant topic percentage shows that value in a more readable form. The category or theme labels were constructed as guidance for framing each of the 4 topics. The row order was preserved so mapping is possible. Note: document IDs could be set and defined as an additional traceable indexes.

In [ ]:
# Estimate the topic mixture for each speech
document_topic_probabilities = final_lda_topic_model.transform(final_document_term_matrix)

document_topic_probabilities_df = pd.DataFrame(
    document_topic_probabilities,
    columns=[f"Topic_{i}" for i in range(1, final_k + 1)]
)

# document-level metadata
document_topic_probabilities_df.insert(0, "document", speeches_df["document"])
document_topic_probabilities_df.insert(1, "year", speeches_df["year"])
document_topic_probabilities_df.insert(2, "president", speeches_df["president"])

# Identify the dominant topic for each speech
document_topic_probabilities_df["dominant_topic"] = (
    document_topic_probabilities_df[[f"Topic_{i}" for i in range(1, final_k + 1)]]
    .idxmax(axis=1)
)

document_topic_probabilities_df["dominant_topic_probability"] = (
    document_topic_probabilities_df[[f"Topic_{i}" for i in range(1, final_k + 1)]]
    .max(axis=1)
)

# Add dominant topic probability as a percentage for easier reading
document_topic_probabilities_df["dominant_topic_pct"] = (
    document_topic_probabilities_df["dominant_topic_probability"] * 100
).round(2)

# Create some topic labels for interpretation (made these, can be designed by you)
topic_label_map = {
    "Topic_1": "Constitutional governance and executive power", # some possible descriptors per topic based on term contents
    "Topic_2": "Peace, justice, and world affairs",
    "Topic_3": "Union, states, and public government",
    "Topic_4": "National identity"
}

document_topic_probabilities_df["dominant_topic_label"] = (
    document_topic_probabilities_df["dominant_topic"].map(topic_label_map)
)

# lookup table of top terms for each topic
dominant_topic_top_terms_df = (
    final_topic_term_summary_df.sort_values(["topic", "rank_within_topic"])
    .groupby("topic")["term"]
    .apply(lambda terms: ", ".join(terms.head(5)))
    .reset_index()
)

dominant_topic_top_terms_df["dominant_topic"] = (
    "Topic_" + dominant_topic_top_terms_df["topic"].astype(str)
)

dominant_topic_top_terms_df = dominant_topic_top_terms_df.rename(
    columns={"term": "dominant_topic_top_terms"}
)[["dominant_topic", "dominant_topic_top_terms"]]

# top terms for the dominant topic to each speech
document_topic_probabilities_df = document_topic_probabilities_df.merge(
    dominant_topic_top_terms_df,
    on="dominant_topic",
    how="left"
)

display(document_topic_probabilities_df.head())

Summary table of basic coverage (min, max...)

In [ ]:
topic_probability_columns = [f"Topic_{i}" for i in range(1, final_k + 1)]

topic_weight_sums = final_lda_topic_model.components_.sum(axis=1)
topic_mean_probabilities = document_topic_probabilities_df[topic_probability_columns].mean(axis=0).values
topic_max_probabilities = document_topic_probabilities_df[topic_probability_columns].max(axis=0).values
topic_min_probabilities = document_topic_probabilities_df[topic_probability_columns].min(axis=0).values

final_topic_summary_df = pd.DataFrame({
    "topic": topic_probability_columns,
    "topic_label": [topic_label_map[f"Topic_{i}"] for i in range(1, final_k + 1)],
    "total_topic_weight": topic_weight_sums,
    "mean_document_probability": topic_mean_probabilities,
    "max_document_probability": topic_max_probabilities,
    "min_document_probability": topic_min_probabilities
})

display(final_topic_summary_df)

## 10c. looking across speeches, presidents and document -> topic dominance (estimated)

In [ ]:
document_topic_context_plot = px.scatter(
    document_topic_probabilities_df,
    x="year",
    y="dominant_topic_pct",
    color="dominant_topic",
    hover_data=[
        "document",
        "president",
        "dominant_topic_label",
        "dominant_topic_top_terms"
    ],
    title="Dominant topic strength by speech"
)

document_topic_context_plot.update_layout(
    xaxis_title="Year",
    yaxis_title="Dominant topic (%)"
)

document_topic_context_plot.write_html(
    outputs_dir / "10b_document_topic_context_plot.html"
)

document_topic_context_plot.show()

## 10d. topics by speeches

In [ ]:
dominant_topic_distribution_df = (
    document_topic_probabilities_df["dominant_topic"]
    .value_counts()
    .rename_axis("dominant_topic")
    .reset_index(name="n_speeches")
)

dominant_topic_distribution_df["topic_label"] = (
    dominant_topic_distribution_df["dominant_topic"].map(topic_label_map)
)

display(dominant_topic_distribution_df)

Distributions of topics by speeches..

In [ ]:
for topic_name in [f"Topic_{i}" for i in range(1, final_k + 1)]:
    print(f"\nTop speeches for {topic_name}")
    display(
        document_topic_probabilities_df[
            ["document", "year", "president", topic_name]
        ]
        .sort_values(topic_name, ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

## Conclusions & summary

If you made it this far, well done! You will have learned that analyses are a series of decision points, each of which relate in varying ways to model application, inputs, and then evaluation of outputs based on your starting research question.

Some key things to note, when applying methodology like LDA or similarly any statistical/machine learning framework via a programming language from many of the packages and libraries out there, one might find many options to choose from. Here scikit learn LDA was used, however, for those who work closely in natural language analyses domains or work with text more generally may have employed gensim, or other packages for LDA in python. The theoretical underpinnings of the models under the hood achieve the same goal and function however, the means may be different, in some cases it could be the approach for topic inference from a corpus or parameter options available for better performance. In the model used in this iteration, it uses variational bayes inference principles whilst TopicModels in R uses other approximation methods and sampling approaches. The authors and designers of these complex model packages provide documentation and literature to support the development of the model, these can provide a guide as well as conventions in your given domain.

We were able to:
1. Clean and prepare data (basic)
2. Pre-process text data
3. explore tokenization
4. Explore and apply feature reduction
5. Explore topic selection criteria and metrics
6. Apply LDA models
7. Throughout the workbook we used consistent inspection of dataframes through checks and visualisations


## A list of the tools used with links to documentation

1. [NLTK Toolkit](https://www.nltk.org/)
2. [Scikit Learn - LDA model](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)
3. [Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent dirichlet allocation. Journal of machine Learning research, 3(Jan), 993-1022.](https://www.jmlr.org/papers/volume3/blei03a/blei03a.pdf)
4. [SpaCY & Gensim, python based, for LDA - a brief example](https://towardsdatascience.com/topic-modelling-in-python-with-spacy-and-gensim-dc8f7748bdbf/)

Theoretical reading for LDA

- Blei D.M., Ng A.Y., Jordan M.I. (2003). Latent Dirichlet Allocation. Journal of Machine Learning
Research, 3, 993–1022.
- Phan X.H., Nguyen L.M., Horguchi S. (2008). Learning to Classify Short and Sparse Text & Web
with Hidden Topics from Large-scale Data Collections. In Proceedings of the 17th International
World Wide Web Conference (WWW 2008), pages 91–100, Beijing, China.
- Lu, B., Ott, M., Cardie, C., Tsou, B.K. (2011). Multi-aspect Sentiment Analysis with Topic Models.
In Proceedings of the 2011 IEEE 11th International Conference on Data Mining Workshops, pages
81–88.